<a href="https://colab.research.google.com/github/Zikai-Dou/dataflow-ai-labs/blob/main/Lab2_Transformer_Dataflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Visualizing Transformer Dataflow

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Module**: 2, Dataflow Execution Models  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU or CPU (mostly static analysis)  
**Expected time**: 90–120 minutes


---

## Learning objectives
By the end of this lab you will be able to:
1. Trace a transformer block into an FX graph with torch.fx.symbolic_trace.
2. Annotate every node with FLOPs and bytes and compute its arithmetic intensity.
3. Classify nodes as compute-bound / memory-bound against your device's true ridge.
4. Predict which nodes become scheduling bottlenecks on a dataflow accelerator.

## Prerequisites
Lab 1 complete, plus the Module 2 lecture on dataflow execution and FX graphs.

## Working through this lab
1. Run the notebook cells in order.
2. Complete each learner task before opening its **Hint**.
3. If you get stuck, expand the **Hint** to view the full reference solution from the authoring notebook.
4. Run each **Self-check** cell after completing the corresponding task.



## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


## Part 0: Setup

`torch.fx` is PyTorch's mechanism for capturing models as graphs. It works on *symbolic-traceable* modules; roughly, modules whose `forward` doesn't contain data-dependent Python control flow.

Modern LLMs like the full Qwen 2.5 model *are not* cleanly fx-traceable because of KV cache handling, position embedding math, and generation loops. For this lab we use a **simplified but faithful transformer block** that mirrors Qwen's structure (pre-norm, Q/K/V projections, scaled-dot-product attention, SwiGLU-style FFN) while remaining fully traceable.

> In Module 3 you'll see `torch.export` and `torch.compile`'s Dynamo, which *do* handle the full model. FX is the pedagogically cleaner starting point.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.fx
from collections import OrderedDict

torch.manual_seed(0)

print(f"torch = {torch.__version__}")
print(f"device = {'cuda' if torch.cuda.is_available() else 'cpu'}")


> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


In [ ]:
# Shared hardware reference. identical across every lab in this course.
# ridge = peak FP16 TFLOPS / peak memory bandwidth (TB/s): the arithmetic intensity
# (FLOPs/byte) where a workload flips from memory-bound to compute-bound. These match
# the lecture decks (T4 ridge ~203, H100 ~295).
HW_SPECS = {
    'Tesla T4':              {'fp16_tflops': 65.0,  'bw_tb_s': 0.320, 'ridge': 203},
    'NVIDIA A100-SXM4-40GB': {'fp16_tflops': 312.0, 'bw_tb_s': 1.555, 'ridge': 201},
    'NVIDIA A100-SXM4-80GB': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039, 'ridge': 153},
    'NVIDIA H100':           {'fp16_tflops': 989.0, 'bw_tb_s': 3.35,  'ridge': 295},
}
# IBM Spyre (confirm against the current spec sheet before publishing):
SPYRE_SPECS = {'fp16_tflops': 98.0, 'lpddr5_tb_s': 0.204, 'scratchpad_mb': 64, 'scratchpad_tb_s': 10.0}

def lookup_hw(name):
    for _k, _v in HW_SPECS.items():
        if _k in name:
            return _k, _v
    return 'Tesla T4', HW_SPECS['Tesla T4']   # Colab free-tier default


In [ ]:
class TinyTransformerBlock(nn.Module):
    """
    A traceable transformer block. Dimensions roughly match Qwen 2.5 0.5B:
        hidden_dim = 896, num_heads = 14, head_dim = 64, ffn_dim = 4864.

    For lab tractability we use smaller dims by default. The ratios are preserved
    so the FLOP / memory analysis transfers directly to the real model.

    Architecture: pre-norm, scaled-dot-product attention (no GQA for simplicity),
    SwiGLU-adjacent FFN (gated), residual adds.
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.ffn_dim    = ffn_dim

        # Attention projections
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Norms
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

        # FFN: gated ("SwiGLU-ish"; two up-projections, one down)
        self.ffn_gate = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_up   = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x):
        # x: [batch, seq, hidden]
        B, S, H = x.shape[0], x.shape[1], x.shape[2]
        D = self.head_dim
        N = self.num_heads

        # --- Attention block (pre-norm) ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, N, D).transpose(1, 2)   # [B, N, S, D]
        k = self.k_proj(h).reshape(B, S, N, D).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, N, D).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(D)  # [B, N, S, S]
        attn   = torch.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)                               # [B, N, S, D]
        out    = out.transpose(1, 2).reshape(B, S, H)
        out    = self.o_proj(out)
        x      = x + out

        # --- FFN block (pre-norm, gated) ---
        h = self.ln2(x)
        gate = torch.nn.functional.silu(self.ffn_gate(h))
        up   = self.ffn_up(h)
        h    = gate * up                                              # gated activation
        h    = self.ffn_down(h)
        x    = x + h
        return x


# Instantiate and smoke-test
HIDDEN_DIM = 512
NUM_HEADS  = 8
FFN_DIM    = 2048
BATCH      = 2
SEQ        = 128

block = TinyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM)
out   = block(dummy)
assert out.shape == dummy.shape
print(f"Block output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")


## Part 1: Trace the block with `torch.fx`

`torch.fx.symbolic_trace` walks through `forward()` with *proxy* tensors, recording every operation that touches a proxy. The output is a `GraphModule` containing a `Graph` of `Node` objects.

Each node has:
- `node.op`; one of `placeholder`, `call_function`, `call_method`, `call_module`, `output`.
- `node.target`: the function/method/module being called.
- `node.args`, `node.kwargs`; its inputs.
- `node.meta`: a dict where you can attach your own metadata (we'll use this in Part 3).

**Your task**:
1. Trace `block` into a `GraphModule`.
2. Print the tabular representation of the graph.
3. Count how many nodes fall into each `op` category.


In [ ]:
# 1.1: trace
# 1.3: count by op type

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 1.1: trace
gm = torch.fx.symbolic_trace(block)

# 1.2: tabular view (useful for reference; truncated)
gm.graph.print_tabular()

# 1.3: count by op type
op_counts = {}
for node in gm.graph.nodes:
    op_counts[node.op] = op_counts.get(node.op, 0) + 1

print("\nNode counts by op type:")
for k, v in op_counts.items():
    print(f"  {k:20s}: {v}")
print(f"  total: {sum(op_counts.values())}")
```

</details>


In [ ]:
# --- Self-check 1 -------------------------------------------------------
assert isinstance(gm, torch.fx.GraphModule), "gm must be a GraphModule from torch.fx.symbolic_trace"
assert "placeholder" in op_counts and op_counts["placeholder"] >= 1
assert "output" in op_counts and op_counts["output"] == 1
assert "call_module" in op_counts, "Should have call_module nodes (Linears, LayerNorms)"
assert op_counts["call_module"] >= 8, "Expected >= 8 submodule calls (q/k/v/o/ln1/ln2/ffn_gate/ffn_up/ffn_down)"
total = sum(op_counts.values())
assert 20 <= total <= 60, f"Graph size looks off: {total} nodes"
print(f"[OK] Traced block into a {total}-node graph.")


## Part 2: Visualize the graph

Reading the tabular print is fine for debugging but doesn't give you a feel for the *dataflow shape*. Let's draw it.

We'll use `networkx` (ships with Colab) to lay out the graph and `matplotlib` to render it. We color nodes by `op` type so `call_module` (parameterized ops like Linear) pop visually; these are the nodes the dataflow compiler will spend the most effort on.


In [ ]:
# 2.1: build networkx DiGraph from FX
# 2.2: edge count

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# 2.1: build networkx DiGraph from FX
G = nx.DiGraph()
for node in gm.graph.nodes:
    G.add_node(node.name)
for node in gm.graph.nodes:
    for inp in node.all_input_nodes:
        G.add_edge(inp.name, node.name)

TERRACOTTA = "#B85C2C"
CREAM      = "#FAF6F2"
SECONDARY  = "#5C5C5C"

color_for_op = {
    "placeholder":    "#A8C5A8",
    "call_module":    TERRACOTTA,
    "call_function":  "#5C5C5C",
    "call_method":    "#BFBFBF",
    "output":         "#2F2F2F",
    "get_attr":       "#CDCDCD",
}
op_of  = {n.name: n.op for n in gm.graph.nodes}
labels = {n.name: n.name for n in gm.graph.nodes}
colors = [color_for_op.get(op_of[name], "#999") for name in G.nodes]

try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    pos = nx.spring_layout(G, seed=0, k=0.9, iterations=80)

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=SECONDARY, arrows=True, arrowsize=10, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7)
ax.set_title("TinyTransformerBlock; FX dataflow graph", fontsize=14, color="#1A1A1A")
ax.set_facecolor(CREAM)
ax.axis("off")

legend_handles = [Patch(facecolor=c, label=k) for k, c in color_for_op.items()]
ax.legend(handles=legend_handles, loc="lower left", frameon=False)
plt.tight_layout()
plt.show()

# 2.2: edge count
n_edges = G.number_of_edges()
print(f"Graph has {G.number_of_nodes()} nodes and {n_edges} edges.")
```

</details>


In [ ]:
# --- Self-check 2 -------------------------------------------------------
assert G.number_of_nodes() == sum(op_counts.values())
assert n_edges is not None and n_edges >= G.number_of_nodes() - 1
# Sanity: the output should have at least one incoming edge.
out_nodes = [n for n in gm.graph.nodes if n.op == "output"]
assert len(out_nodes) == 1
print(f"[OK] Graph visualized. {G.number_of_nodes()} nodes, {n_edges} edges.")


## Part 3: Annotate every node with compute cost and memory footprint

A dataflow compiler needs two numbers for every node to make scheduling and tiling decisions:
- **FLOPs**: how much compute does this node do?
- **Bytes moved**: how much data must flow across the inputs and output?

Their ratio; *arithmetic intensity* (AI); is the most important single number in performance analysis. `AI = FLOPs / bytes`. A node with AI > 10 is likely compute-bound; AI < 1 is memory-bound.

We'll do a simple first-order model:

| Op | FLOPs | Bytes (fp16) |
|---|---|---|
| `nn.Linear(d_in, d_out)` on `[B, S, d_in]` | `2 * B * S * d_in * d_out` | `2 * (B*S*d_in + d_in*d_out + B*S*d_out)` |
| `matmul([B,N,S,D] @ [B,N,D,S])` (Q·Kᵀ) | `2 * B * N * S * S * D` | `2 * (B*N*S*D + B*N*D*S + B*N*S*S)` |
| `matmul([B,N,S,S] @ [B,N,S,D])` (attn·V) | `2 * B * N * S * S * D` | `2 * (B*N*S*S + B*N*S*D + B*N*S*D)` |
| `LayerNorm`, `softmax`, `silu`, `mul`, `add` | ~0 FLOPs (relative), bytes = 2 × elementwise tensor size | (memory-bound) |

**Your task**: for each node in `gm.graph`, attach `node.meta["flops"]` and `node.meta["bytes"]` using the helper functions provided.


In [ ]:
    # default: elementwise / view / reshape / transpose / softmax / silu / mul / add / truediv
# Print table

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
BYTES_PER_ELEM = 2  # fp16

def linear_flops(in_features, out_features, batch_seq):
    return 2 * batch_seq * in_features * out_features

def linear_bytes(in_features, out_features, batch_seq):
    elems = batch_seq * in_features + in_features * out_features + batch_seq * out_features
    return BYTES_PER_ELEM * elems

def matmul_flops(a_shape, b_shape):
    bat = 1
    for d in a_shape[:-2]:
        bat *= d
    M, K = a_shape[-2], a_shape[-1]
    N    = b_shape[-1]
    return 2 * bat * M * N * K

def matmul_bytes(a_shape, b_shape, out_shape):
    def prod(s):
        p = 1
        for d in s: p *= d
        return p
    return BYTES_PER_ELEM * (prod(a_shape) + prod(b_shape) + prod(out_shape))

def elementwise_bytes(shape):
    p = 1
    for d in shape: p *= d
    return BYTES_PER_ELEM * 2 * p

from torch.fx.passes.shape_prop import ShapeProp
ShapeProp(gm).propagate(dummy)

def shape_of(node):
    return list(node.meta["tensor_meta"].shape) if "tensor_meta" in node.meta else None

for node in gm.graph.nodes:
    node.meta["flops"] = 0
    node.meta["bytes"] = 0
    out_shape = shape_of(node)

    if node.op in ("placeholder", "output", "get_attr"):
        continue

    if node.op == "call_module":
        sub = gm.get_submodule(node.target)
        if isinstance(sub, nn.Linear):
            in_shape = shape_of(node.all_input_nodes[0])
            batch_seq = 1
            for d in in_shape[:-1]:
                batch_seq *= d
            node.meta["flops"] = linear_flops(sub.in_features, sub.out_features, batch_seq)
            node.meta["bytes"] = linear_bytes(sub.in_features, sub.out_features, batch_seq)
        elif isinstance(sub, nn.LayerNorm):
            node.meta["flops"] = 0  # small, ignore for first-order model
            node.meta["bytes"] = elementwise_bytes(out_shape)
        else:
            node.meta["bytes"] = elementwise_bytes(out_shape) if out_shape else 0
        continue

    if node.op == "call_function" and node.target is torch.matmul:
        a_shape = shape_of(node.all_input_nodes[0])
        b_shape = shape_of(node.all_input_nodes[1])
        node.meta["flops"] = matmul_flops(a_shape, b_shape)
        node.meta["bytes"] = matmul_bytes(a_shape, b_shape, out_shape)
        continue

    # default: elementwise / view / reshape / transpose / softmax / silu / mul / add / truediv
    if out_shape is not None:
        node.meta["bytes"] = elementwise_bytes(out_shape)

# Print table
print(f"{'name':30s} {'op':15s} {'flops':>14s} {'bytes':>14s} {'AI':>8s}")
print("-" * 85)
for node in gm.graph.nodes:
    f = node.meta["flops"]
    b = node.meta["bytes"]
    ai = (f / b) if b > 0 else 0.0
    print(f"{node.name:30s} {node.op:15s} {f:14,d} {b:14,d} {ai:8.2f}")
```

</details>


In [ ]:
# --- Self-check 3 ------------------------------------------------------
# Every node should have flops and bytes keys set.
assert all("flops" in n.meta and "bytes" in n.meta for n in gm.graph.nodes)

# Total FLOPs should be dominated by the matmuls and Linears.
total_flops = sum(n.meta["flops"] for n in gm.graph.nodes)
total_bytes = sum(n.meta["bytes"] for n in gm.graph.nodes)

# Rough expected FLOP count for this block at HIDDEN=512 NUM_HEADS=8 FFN=2048 B=2 S=128
# (batch_seq = B*S = 256):
#   4 QKVO Linears : 4 * (2 * 256 * 512 * 512)                ~ 5.37e8
#   2 attn matmuls : 2 * (2 * 2 * 8 * 128 * 128 * 64)         ~ 6.71e7
#   3 FFN Linears  : (gate) 2*256*512*2048 + (up) 2*256*512*2048
#                    + (down) 2*256*2048*512                  ~ 1.61e9
#   total                                                     ~ 2.2e9
assert 5e8 < total_flops < 5e9, f"Total FLOPs look off: {total_flops:,}"
assert total_bytes > 0
print(f"[OK] Total FLOPs: {total_flops:,}  |  Total bytes: {total_bytes:,}")


## Part 4 - Classify each node

The arithmetic-intensity **ridge** -- where a workload flips from memory-bound to compute-bound -- is `peak_compute / peak_bandwidth`, your device's real ridge (about **203 FLOPs/byte on a T4**, **295 on an H100**). We classify each node against it:

- **compute-bound**  if `AI >= ridge`
- **memory-bound**   if `0 < AI < ridge`
- **cheap**          if `bytes < 1e4` (tiny tensors -- noise)

At this toy block size you will find most GEMMs land **below** the ridge -- they are memory-bound even though they are matmuls. That is exactly the Module 6 lesson: the ridge is high, so you have to be huge to be compute-bound. Later Modules revisit the threshold for Spyre (scratchpad shifts the ridge) and TPU (VMEM + HBM give two ridges).


In [ ]:
# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
_hwname, _hw = lookup_hw(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tesla T4")
GPU_RIDGE_AI = _hw["ridge"]
print(f"Classifying against {_hwname} ridge = {GPU_RIDGE_AI} FLOPs/byte")
CHEAP_BYTES  = 1e4

class_counts = {"compute": 0, "memory": 0, "cheap": 0}
for node in gm.graph.nodes:
    b = node.meta["bytes"]
    f = node.meta["flops"]
    if b < CHEAP_BYTES:
        cls = "cheap"
    else:
        ai = f / b if b > 0 else 0.0
        cls = "compute" if ai >= GPU_RIDGE_AI else "memory"
    node.meta["class"] = cls
    class_counts[cls] += 1

print("Node classification:")
for k, v in class_counts.items():
    print(f"  {k:10s}: {v}")
```

</details>


In [ ]:
# Re-draw the graph colored by classification (instead of op type)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Re-draw the graph colored by classification (instead of op type)
class_color = {
    "compute": TERRACOTTA,    # heavy lifting
    "memory":  "#5C5C5C",     # memory-bound operators
    "cheap":   "#CDC5BA",     # low-cost
}
class_of = {n.name: n.meta.get("class", "cheap") for n in gm.graph.nodes}
colors2  = [class_color[class_of[name]] for name in G.nodes]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=SECONDARY, arrows=True, arrowsize=10, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors2, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7)
ax.set_title("Classification: compute-bound (terracotta) vs memory-bound (gray) vs cheap",
             fontsize=14, color="#1A1A1A")
ax.set_facecolor(CREAM)
ax.axis("off")
legend_handles = [Patch(facecolor=c, label=k) for k, c in class_color.items()]
ax.legend(handles=legend_handles, loc="lower left", frameon=False)
plt.tight_layout()
plt.show()
```

</details>


In [ ]:
# --- Self-check 4 ------------------------------------------------------
assert sum(class_counts.values()) == sum(op_counts.values()), \
    "Every node must be classified exactly once."
# Sanity: you should have at least a few compute-bound nodes (the Linears, matmuls).
# At the true device ridge (~203 on a T4) most ops in this toy block are memory-bound --
# the Module 6 punchline arriving early. Require only a valid partition + memory-bound nodes.
assert class_counts["memory"] >= 2, f"Expected memory-bound nodes; got {class_counts}"
# ... and some memory-bound / cheap ones (softmax, elementwise mul, add, etc).
assert class_counts["compute"] + class_counts["memory"] + class_counts["cheap"] == sum(op_counts.values())
print(f"[OK] Classification complete: {class_counts}")


## Part 5: Analysis and Interpretation

Use your traced graph, node annotations, and arithmetic-intensity classifications to identify likely bottlenecks and explain how they may behave on a dataflow target.

> Write your analysis before opening the Hint below.


<details>
<summary><strong>Hint</strong></summary>

### 5.1: Three likely bottleneck nodes on a dataflow target

Reference picks (node names will vary slightly with trace):

1. **`transpose` / `reshape` nodes around Q, K, V** (e.g. `transpose`, `reshape`, `transpose_1`, `transpose_2`). On a GPU these are metadata ops. On Spyre / TPU / Groq they frequently require physical data movement because the SRAM tiling and the logical `[B, N, S, D]` layout are not aligned. The Spyre compiler reports this as **T_shuffle** overhead.

2. **`softmax`** on the attention scores. A softmax along the last axis is a reduction (subtract max, exp, sum, divide. When the attention matrix is tiled across SRAM banks, the reduction must serialize waiting for partial sums from other banks. This is why FlashAttention-style fused kernels exist: they compute softmax *while* streaming the QKᵀ tiles.

3. **The add nodes for residual connections** (e.g. `add`, `add_1`). Individually cheap, but they *synchronize* two long paths in the graph; the attention branch and the residual bypass. Whenever a dataflow scheduler needs to rendezvous two paths of different length, it stalls the shorter one. On a GPU the L2 cache absorbs this; on a pure SRAM dataflow target it can dominate the gap.

### 5.2: Fusion opportunity

The gated-FFN sequence is the canonical fusion target:

```
ln2 -> ffn_gate -> silu -> mul (with ffn_up) -> ffn_down
```

A fused node would read the input once, hold the pre-gate and up-proj intermediate in registers/SRAM, apply SiLU and the gating multiply without writing back to device memory, and then do the down-projection. This converts roughly 5 device-memory round-trips into 1. On a GPU, TorchInductor will produce a fused triton kernel for this. On TPU, XLA does the same via HLO fusion passes. On Spyre, the graph IR explicitly represents this as a single fused operator.

An excellent answer would note that **ffn_down can't always be fused in**: it has a large matmul inside, and keeping all of FFN in SRAM costs too much buffer. So the realistic fusion is `ln2 -> ffn_gate -> silu -> mul_with_ffn_up` as one fused op, with `ffn_down` following.

### 5.3: Arithmetic intensity of QKᵀ at longer sequence

At our lab values (B=2, N=8, S=128, D=64):
- FLOPs = 2 · B · N · S · S · D = 2·2·8·128·128·64 ≈ 3.36e7
- Bytes ≈ 2 · (B·N·S·D + B·N·D·S + B·N·S·S) = 2·(131072 + 131072 + 262144) ≈ 1.05e6
- AI ≈ **32 FLOPs/byte**: below the T4 ridge (~203), so memory-bound at this size.

At S=4096 (same B, N, D):
- FLOPs = 2·2·8·4096·4096·64 ≈ 3.44e10   (1024× more)
- Bytes ≈ 2 · (B·N·S·D + B·N·D·S + B·N·S·S) = 2·(4194304 + 4194304 + 268435456) ≈ 5.53e8
- AI ≈ **62 FLOPs/byte**: higher, but still far below the ridge, so still memory-bound.

**Why the counter-intuitive result**: the attention matrix itself (B·N·S·S) grows as S², but so does the compute (S²·D). They scale together, with D as the constant. The *real* problem at long sequence isn't arithmetic intensity; it's **total memory**. The attention matrix alone at S=4096 is 256 MB in fp16. This is why FlashAttention is critical: it avoids materializing the full attention matrix, trading extra compute for a much smaller memory footprint.

A strong answer flags the distinction between **AI** (a compute/bandwidth ratio) and **absolute memory pressure** (a capacity constraint). They're not the same limit.

</details>


## Part 6: Update the Landscape Map

Module 2 adds a new row to the shared Accelerator Landscape Map: **"Graph representation complexity"**: how much work does each system's compiler do to get from the source PyTorch graph to something executable?

Fill in the row for PyTorch-on-GPU based on what you observed. We will fill in other accelerators in Module 3–4.


In [ ]:
# 6.1: Landscape Map row for the runtime you measured.
# Build a clean accelerator label so a CPU runtime doesn't render as "GPU (CPU)".
    # This lab only *captures* the FX graph. symbolic_trace does not compile, fuse, or

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 6.1: Landscape Map row for the runtime you measured.
# Build a clean accelerator label so a CPU runtime doesn't render as "GPU (CPU)".
accelerator_name = (
    f"GPU ({torch.cuda.get_device_name(0)})"
    if torch.cuda.is_available()
    else "CPU runtime"
)

landscape_row_w2 = {
    "accelerator":              accelerator_name,
    "graph_capture_mechanism":  "torch.fx symbolic_trace (static) / TorchDynamo (production)",
    "graph_size":               f"{G.number_of_nodes()} nodes / {n_edges} edges",
    # This lab only *captures* the FX graph. symbolic_trace does not compile, fuse, or
    # generate kernels. That is torch.compile / TorchInductor territory.
    "compile_time_work":        ("FX graph capture only in this lab; with torch.compile, "
                                 "TorchInductor may perform fusion and kernel code generation"),
    "scheduling_granularity":   "kernel-level once compiled (operators fused into CUDA kernels)",
    "compute_bound_nodes":      class_counts["compute"],
    "memory_bound_nodes":       class_counts["memory"],
}

for k, v in landscape_row_w2.items():
    print(f"  {k:28s}: {v}")
```

</details>


In [ ]:
# --- Self-check 5 ------------------------------------------------------
required = {"accelerator", "graph_capture_mechanism", "graph_size",
            "compile_time_work", "scheduling_granularity",
            "compute_bound_nodes", "memory_bound_nodes"}
assert set(landscape_row_w2.keys()) == required
assert all(v is not None for v in landscape_row_w2.values())
print("[OK] Module 2 Landscape Map row complete.")
